<a href="https://www.kaggle.com/code/ameythakur20/customer-churn-prediction-121fe-15cv-triple-ensemble" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Churn Prediction: 121-Feature Triple Boosting Ensemble

**Playground Series, Season 6, Episode 3**

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

Predicting customer attrition requires capturing high-order interactions between service
usage patterns and billing lifecycle metrics. This analytical guide utilizes 121 engineered
features across a 15-fold stratified cross-validation strategy. The final predictive
vector is synthesized through a triple-boosting ensemble (XGBoost, CatBoost, and LightGBM),
blended via the COBYLA optimization algorithm to maximize discriminative power.

**Outline:**

1. [Configuration and Environment](#1-configuration-and-environment)
2. [Data Acquisition and Integration](#2-data-acquisition-and-integration)
3. [Empirical Data Analysis](#3-empirical-data-analysis)
4. [High-Fidelity Feature Engineering](#4-high-fidelity-feature-engineering)
5. [Primary Model Training Logic](#5-primary-model-training-logic)
6. [Triple Ensemble Blending](#6-triple-ensemble-blending)
7. [Predictive Outcome Analysis](#7-predictive-outcome-analysis)
8. [Submission Generation](#8-submission-generation)

---

## 1. Configuration and Environment

All hyperparameters, paths, and environment configurations are centralized here. A
15-fold stratified cross-validation strategy is employed to generate robust estimates
while remaining within the 180-minute computational budget for the triple ensemble.

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import warnings
import gc
import time
from itertools import combinations

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from scipy.stats import rankdata
from scipy.optimize import minimize, Bounds
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb_mod
from catboost import CatBoostClassifier

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")

In [ ]:
# Configuration
class CFG:
    TARGET = 'Churn'
    N_FOLDS = 15       # Outer CV folds
    INNER_FOLDS = 5    # Inner CV folds for target encoding
    RANDOM_SEED = 42
    
    # Data paths (update these paths as needed)
    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
    ORIGINAL_PATH = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Top categoricals for bi-gram/tri-gram combinations (by feature importance)
TOP_CATS_FOR_NGRAM = [
    'Contract', 'InternetService', 'PaymentMethod',
    'OnlineSecurity', 'TechSupport', 'PaperlessBilling'
]

XGB_PARAMS = {
    'n_estimators': 30000,
    'learning_rate': 0.0063,
    'max_depth': 5,
    'subsample': 0.81,
    'colsample_bytree': 0.32,
    'min_child_weight': 6,
    'reg_alpha': 3.5017,
    'reg_lambda': 1.2925,
    'gamma': 0.790,
    'random_state': CFG.RANDOM_SEED,
    'early_stopping_rounds': 500,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'enable_categorical': True,
    'device': 'cuda',
    'verbosity': 0,
}

CB_PARAMS = {
    'n_estimators'         : 50000,
    'learning_rate'        : 0.05,
    'eval_metric'          : 'AUC',
    'max_depth'            : 5,
    'auto_class_weights'   : 'Balanced',    
    'random_state'         : CFG.RANDOM_SEED,
    'early_stopping_rounds': 100,
    'verbose'              : False
}


LGB_PARAMS = {
    'n_estimators': 30000,
    'learning_rate': 0.01,
    'max_depth': 6,
    'num_leaves': 63,
    'subsample': 0.7,
    'colsample_bytree': 0.4,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': CFG.RANDOM_SEED,
    'verbose': -1,
    'n_jobs': -1,
}

models = {'XGB': XGBClassifier(**XGB_PARAMS), 'CB': CatBoostClassifier(**CB_PARAMS), 'LGB': LGBMClassifier(**LGB_PARAMS)}

print("Configuration set!")

## 2. Data Acquisition

The competition set is synthetically generated from the original Telco Customer Churn
dataset. Integrating the original provides a broader distribution of real-world patterns
that synthetic generation may not perfectly replicate. This consistently improves
generalization across multiple model types.

In [ ]:
# Load Data
print("Loading datasets...")
train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
orig = pd.read_csv(CFG.ORIGINAL_PATH)

# Encode target variable
train[CFG.TARGET] = train[CFG.TARGET].map({'No': 0, 'Yes': 1}).astype(int)
orig[CFG.TARGET] = orig[CFG.TARGET].map({'No': 0, 'Yes': 1}).astype(int)

# Handle TotalCharges in original data
orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
orig['TotalCharges'].fillna(orig['TotalCharges'].median(), inplace=True)

# Remove customer ID if present
if 'customerID' in orig.columns:
    orig.drop(columns=['customerID'], inplace=True)

# Store IDs for later
train_ids = train['id'].copy()
test_ids = test['id'].copy()

print(f"Train Shape : {train.shape}")
print(f"Test Shape  : {test.shape}")
print(f"Original Shape: {orig.shape}")

## 3. Exploratory Analysis

Profiling the data before engineering features reveals the degree of class imbalance,
the cardinality of categorical columns, and distribution properties of the numerical
fields. These observations directly inform the encoding and interaction strategies
in the subsequent stage.

In [ ]:
# Dataset Overview
print("="*60)
print("TRAINING DATA OVERVIEW")
print("="*60)
display(train.head())

print("\n" + "="*60)
print("DATA TYPES")
print("="*60)
print(train.dtypes.to_frame('dtype'))

In [ ]:
# Missing Values Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set missing values
train_missing = train.isnull().sum()
train_missing = train_missing[train_missing > 0]
if len(train_missing) > 0:
    axes[0].barh(train_missing.index, train_missing.values, color='coral')
    axes[0].set_xlabel('Missing Count')
    axes[0].set_title('Missing Values in Training Set')
else:
    axes[0].text(0.5, 0.5, 'No Missing Values!', ha='center', va='center', fontsize=14)
    axes[0].set_title('Missing Values in Training Set')

# Numerical statistics
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
stats_df = train[num_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
axes[1].axis('off')
table = axes[1].table(cellText=stats_df.round(2).values,
                       rowLabels=stats_df.index,
                       colLabels=stats_df.columns,
                       cellLoc='center',
                       loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.5)
axes[1].set_title('Numerical Features Statistics', fontsize=12, pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Target Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Training set target
train_churn = train[CFG.TARGET].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(train_churn.values, labels=['No Churn', 'Churn'], autopct='%1.1f%%', 
            colors=colors, explode=[0, 0.05], startangle=90)
axes[0].set_title('Training Set - Churn Distribution')

# Original dataset target
orig_churn = orig[CFG.TARGET].value_counts()
axes[1].pie(orig_churn.values, labels=['No Churn', 'Churn'], autopct='%1.1f%%', 
            colors=colors, explode=[0, 0.05], startangle=90)
axes[1].set_title('Original Dataset - Churn Distribution')

# Comparison bar chart
x = np.arange(2)
width = 0.35
bars1 = axes[2].bar(x - width/2, [train_churn[0], orig_churn[0]], width, label='No Churn', color='#2ecc71')
bars2 = axes[2].bar(x + width/2, [train_churn[1], orig_churn[1]], width, label='Churn', color='#e74c3c')
axes[2].set_xticks(x)
axes[2].set_xticklabels(['Training', 'Original'])
axes[2].set_ylabel('Count')
axes[2].set_title('Target Distribution Comparison')
axes[2].legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[2].annotate(f'{int(height)}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nChurn Rate - Training: {train[CFG.TARGET].mean()*100:.2f}%")
print(f"Churn Rate - Original: {orig[CFG.TARGET].mean()*100:.2f}%")

In [ ]:
# Numerical Features Distribution by Churn
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(['tenure', 'MonthlyCharges', 'TotalCharges']):
    for churn_val, label, color in [(0, 'No Churn', '#2ecc71'), (1, 'Churn', '#e74c3c')]:
        data = train[train[CFG.TARGET] == churn_val][col]
        axes[idx].hist(data, bins=30, alpha=0.6, label=label, color=color, density=True)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Density')
    axes[idx].set_title(f'{col} Distribution by Churn')
    axes[idx].legend()

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("  - Tenure: Churners tend to have shorter tenure")
print("  - MonthlyCharges: Churners often have higher monthly charges")
print("  - TotalCharges: Churners typically have lower total charges (correlated with tenure)")

In [ ]:
# Categorical Features Analysis
CATS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for idx, col in enumerate(CATS):
    # Calculate churn rate by category
    churn_rate = train.groupby(col)[CFG.TARGET].mean().sort_values(ascending=False)
    
    colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(churn_rate)))
    bars = axes[idx].bar(range(len(churn_rate)), churn_rate.values, color=colors)
    axes[idx].set_xticks(range(len(churn_rate)))
    axes[idx].set_xticklabels(churn_rate.index, rotation=45, ha='right', fontsize=8)
    axes[idx].set_ylabel('Churn Rate')
    axes[idx].set_title(f'{col}', fontsize=10)
    axes[idx].axhline(y=train[CFG.TARGET].mean(), color='black', linestyle='--', alpha=0.5)
    
    # Add value labels
    for bar, val in zip(bars, churn_rate.values):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                      f'{val:.2f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Churn Rate by Categorical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Matrix for Numerical Features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Numerical correlation
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
corr_matrix = train[num_cols + [CFG.TARGET]].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0, ax=axes[0], 
            fmt='.3f', square=True)
axes[0].set_title('Numerical Features Correlation')

# Contract vs InternetService churn rates (important interaction)
pivot = train.pivot_table(values=CFG.TARGET, index='Contract', 
                          columns='InternetService', aggfunc='mean')
sns.heatmap(pivot, annot=True, cmap='RdYlGn_r', center=0.26, ax=axes[1], 
            fmt='.3f', square=True)
axes[1].set_title('Churn Rate: Contract × InternetService Interaction')

plt.tight_layout()
plt.show()

print("\nNote: strong> interaction between Contract type and InternetService!")
print("This justifies using bi-gram/tri-gram composite features.")

## 4. Feature Synthesis

The competitive advantage of this pipeline comes from multi-level feature construction:

- **Frequency encoding** surfaces how common a given value is across the population.
  Rare values tend to exhibit different churn propensities than dense clusters.
- **Arithmetic ratios** between charges and tenure isolate billing consistency
  from absolute magnitude.
- **Service density** compresses sparse binary columns into a single utilization measure.
- **ORIG_proba** maps each feature value to its churn probability in the original dataset,
  injecting real-world prior knowledge into the synthetic feature space.
- **Distribution features** (percentile ranks, z-scores) position each record relative
  to the churner and non-churner populations. This relative positioning is what tree
  models respond to most effectively.
- **Digit properties** exploit the synthetic generation process. First digits, remainders,
  and rounding deviations can leak information about how numerical values were constructed.
- **N-gram composites** (bi-grams, tri-grams) from top categorical features capture joint
  states that individual columns cannot represent alone.

In [ ]:
# Define Feature Groups
CATS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]
NUMS = ['tenure', 'MonthlyCharges', 'TotalCharges']

NEW_NUMS = []
NUM_AS_CAT = []

print("Feature Engineering Pipeline Started...")
print("="*60)

In [ ]:
# 1. Frequency Encoding
print("[1/7] Creating Frequency Encoding features...")
for col in NUMS:
    freq = pd.concat([train[col], orig[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{col}'] = df[col].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{col}')
print(f"    Created {len(NUMS)} frequency features")

In [ ]:
# 2. Arithmetic Interactions
print("[2/7] Creating Arithmetic Interaction features...")
for df in [train, test, orig]:
    df['charges_deviation'] = (df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']).astype('float32')
    df['monthly_to_total_ratio'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1)).astype('float32')
    df['avg_monthly_charges'] = (df['TotalCharges'] / (df['tenure'] + 1)).astype('float32')
NEW_NUMS += ['charges_deviation', 'monthly_to_total_ratio', 'avg_monthly_charges']
print(f"    Created 3 arithmetic features")

In [ ]:
# 3. Service Counts
print("[3/7] Creating Service Count features...")
SERVICE_COLS = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for df in [train, test, orig]:
    df['service_count'] = (df[SERVICE_COLS] == 'Yes').sum(axis=1).astype('float32')
    df['has_internet'] = (df['InternetService'] != 'No').astype('float32')
    df['has_phone'] = (df['PhoneService'] == 'Yes').astype('float32')
NEW_NUMS += ['service_count', 'has_internet', 'has_phone']
print(f"    Created 3 service count features")

In [ ]:
# 4. ORIG_proba Mapping
print("[4/7] Creating ORIG_proba features (target probability from original data)...")
for col in CATS + NUMS:
    tmp = orig.groupby(col)[CFG.TARGET].mean()
    _name = f"ORIG_proba_{col}"
    train = train.merge(tmp.rename(_name), on=col, how="left")
    test = test.merge(tmp.rename(_name), on=col, how="left")
    for df in [train, test]:
        df[_name] = df[_name].fillna(0.5).astype('float32')
    NEW_NUMS.append(_name)
print(f"    Created {len(CATS + NUMS)} ORIG_proba features")

In [ ]:
# 5. Distribution Features
print("[5/7] Creating Distribution Features (percentile ranks, z-scores)...")

def pctrank_against(values, reference):
    ref_sorted = np.sort(reference)
    return (np.searchsorted(ref_sorted, values) / len(ref_sorted)).astype('float32')

def zscore_against(values, reference):
    mu, sigma = np.mean(reference), np.std(reference)
    return (np.zeros(len(values), dtype='float32') if sigma == 0 
            else ((values - mu) / sigma).astype('float32'))

orig_churner_tc    = orig.loc[orig[CFG.TARGET] == 1, 'TotalCharges'].values
orig_nonchurner_tc = orig.loc[orig[CFG.TARGET] == 0, 'TotalCharges'].values
orig_tc            = orig['TotalCharges'].values
orig_is_mc_mean    = orig.groupby('InternetService')['MonthlyCharges'].mean()

for df in [train, test]:
    tc = df['TotalCharges'].values
    df['pctrank_nonchurner_TC']  = pctrank_against(tc, orig_nonchurner_tc)
    df['pctrank_churner_TC']     = pctrank_against(tc, orig_churner_tc)
    df['pctrank_orig_TC']        = pctrank_against(tc, orig_tc)
    df['zscore_churn_gap_TC'] = (np.abs(zscore_against(tc, orig_churner_tc)) - 
                                 np.abs(zscore_against(tc, orig_nonchurner_tc))).astype('float32')
    df['zscore_nonchurner_TC'] = zscore_against(tc, orig_nonchurner_tc)
    df['pctrank_churn_gap_TC'] = (pctrank_against(tc, orig_churner_tc) - 
                                  pctrank_against(tc, orig_nonchurner_tc)).astype('float32')
    df['resid_IS_MC'] = (df['MonthlyCharges'] - df['InternetService'].map(orig_is_mc_mean).fillna(0)).astype('float32')
    
    vals = np.zeros(len(df), dtype='float32')
    for cat_val in orig['InternetService'].unique():
        mask = df['InternetService'] == cat_val
        ref = orig.loc[orig['InternetService'] == cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
    df['cond_pctrank_IS_TC'] = vals
    
    vals = np.zeros(len(df), dtype='float32')
    for cat_val in orig['Contract'].unique():
        mask = df['Contract'] == cat_val
        ref = orig.loc[orig['Contract'] == cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
    df['cond_pctrank_C_TC'] = vals

DIST_FEATURES = [
    'pctrank_nonchurner_TC', 'zscore_churn_gap_TC', 'pctrank_churn_gap_TC',
    'resid_IS_MC', 'cond_pctrank_IS_TC', 'zscore_nonchurner_TC',
    'pctrank_orig_TC', 'pctrank_churner_TC', 'cond_pctrank_C_TC'
]
NEW_NUMS += DIST_FEATURES
print(f"    Created {len(DIST_FEATURES)} distribution features")

In [ ]:
# 6. Quantile Distance Features
print("[6/7] Creating Quantile Distance Features...")
for q_label, q_val in [('q25', 0.25), ('q50', 0.50), ('q75', 0.75)]:
    ch_q = np.quantile(orig_churner_tc, q_val)
    nc_q = np.quantile(orig_nonchurner_tc, q_val)
    for df in [train, test]:
        df[f'dist_To_ch_{q_label}'] = np.abs(df['TotalCharges'] - ch_q).astype('float32')
        df[f'dist_To_nc_{q_label}'] = np.abs(df['TotalCharges'] - nc_q).astype('float32')
        df[f'qdist_gap_To_{q_label}'] = (df[f'dist_To_nc_{q_label}'] - df[f'dist_To_ch_{q_label}']).astype('float32')

QDIST_FEATURES = [
    'qdist_gap_To_q50', 'dist_To_ch_q50', 'dist_To_nc_q50',
    'dist_To_nc_q25', 'qdist_gap_To_q25',
    'dist_To_nc_q75', 'dist_To_ch_q75', 'qdist_gap_To_q75'
]
NEW_NUMS += QDIST_FEATURES
print(f"    Created {len(QDIST_FEATURES)} quantile distance features")

In [ ]:
# 7. Numericals as Categories
print("[7/7] Creating Numericals-as-Categories features...")
for col in NUMS:
    _new = f'CAT_{col}'
    NUM_AS_CAT.append(_new)
    for df in [train, test]:
        df[_new] = df[col].astype(str).astype('category')
print(f"    Created {len(NUMS)} numerical-as-category features")

In [ ]:
# New Digit Features
DIGIT_FEATURES = [
    # Tenure
    'tenure_first_digit','tenure_last_digit','tenure_second_digit',
    'tenure_mod10','tenure_mod12','tenure_num_digits',
    'tenure_is_multiple_10','tenure_rounded_10','tenure_dev_from_round10',

    # MonthlyCharges
    'mc_first_digit','mc_last_digit','mc_second_digit',
    'mc_mod10','mc_mod100','mc_num_digits',
    'mc_is_multiple_10','mc_is_multiple_50',
    'mc_rounded_10','mc_fractional','mc_dev_from_round10',

    # TotalCharges
    'tc_first_digit','tc_last_digit','tc_second_digit',
    'tc_mod10','tc_mod100','tc_num_digits',
    'tc_is_multiple_10','tc_is_multiple_100',
    'tc_rounded_100','tc_fractional','tc_dev_from_round100',

    # Derived
    'tenure_years','tenure_months_in_year',
    'mc_per_digit','tc_per_digit'
]

for df in [train, test]:

    # Tenure digits
    t_str = df['tenure'].astype(str)
    df['tenure_first_digit'] = t_str.str[0].astype(int)
    df['tenure_last_digit'] = t_str.str[-1].astype(int)
    df['tenure_second_digit'] = t_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)

    df['tenure_mod10'] = df['tenure'] % 10
    df['tenure_mod12'] = df['tenure'] % 12
    df['tenure_num_digits'] = t_str.str.len()

    df['tenure_is_multiple_10'] = (df['tenure'] % 10 == 0).astype('float32')

    df['tenure_rounded_10'] = np.round(df['tenure']/10)*10
    df['tenure_dev_from_round10'] = abs(df['tenure'] - df['tenure_rounded_10'])

    # MonthlyCharges
    mc_str = df['MonthlyCharges'].astype(str).str.replace('.', '')

    df['mc_first_digit'] = mc_str.str[0].astype(int)
    df['mc_last_digit'] = mc_str.str[-1].astype(int)
    df['mc_second_digit'] = mc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)

    df['mc_mod10'] = np.floor(df['MonthlyCharges']) % 10
    df['mc_mod100'] = np.floor(df['MonthlyCharges']) % 100

    df['mc_num_digits'] = np.floor(df['MonthlyCharges']).astype(int).astype(str).str.len()

    df['mc_is_multiple_10'] = (np.floor(df['MonthlyCharges']) % 10 == 0).astype('float32')
    df['mc_is_multiple_50'] = (np.floor(df['MonthlyCharges']) % 50 == 0).astype('float32')

    df['mc_rounded_10'] = np.round(df['MonthlyCharges']/10)*10
    df['mc_fractional'] = df['MonthlyCharges'] - np.floor(df['MonthlyCharges'])
    df['mc_dev_from_round10'] = abs(df['MonthlyCharges'] - df['mc_rounded_10'])

    # TotalCharges
    tc_str = df['TotalCharges'].astype(str).str.replace('.', '')

    df['tc_first_digit'] = tc_str.str[0].astype(int)
    df['tc_last_digit'] = tc_str.str[-1].astype(int)
    df['tc_second_digit'] = tc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)

    df['tc_mod10'] = np.floor(df['TotalCharges']) % 10
    df['tc_mod100'] = np.floor(df['TotalCharges']) % 100

    df['tc_num_digits'] = np.floor(df['TotalCharges']).astype(int).astype(str).str.len()

    df['tc_is_multiple_10'] = (np.floor(df['TotalCharges']) % 10 == 0).astype('float32')
    df['tc_is_multiple_100'] = (np.floor(df['TotalCharges']) % 100 == 0).astype('float32')

    df['tc_rounded_100'] = np.round(df['TotalCharges']/100)*100
    df['tc_fractional'] = df['TotalCharges'] - np.floor(df['TotalCharges'])

    df['tc_dev_from_round100'] = abs(df['TotalCharges'] - df['tc_rounded_100'])

    # Derived
    df['tenure_years'] = df['tenure'] // 12
    df['tenure_months_in_year'] = df['tenure'] % 12

    df['mc_per_digit'] = df['MonthlyCharges']/(df['mc_num_digits']+0.001)
    df['tc_per_digit'] = df['TotalCharges']/(df['tc_num_digits']+0.001)

NEW_NUMS += DIGIT_FEATURES
print(f"    Created {len(DIGIT_FEATURES)} digital Numerical features")

In [ ]:
# Create Bi-gram and Tri-gram Composite Categorical Features
print("\n" + "="*60)
print("Creating Bi-gram / Tri-gram Composite Features...")
print("="*60)

BIGRAM_COLS = []
TRIGRAM_COLS = []

# Bi-grams: all pairs from top 6 cats = C(6,2) = 15 pairs
print(f"\nCreating Bi-gram features from top {len(TOP_CATS_FOR_NGRAM)} categoricals:")
print(f"  Columns: {TOP_CATS_FOR_NGRAM}")
for c1, c2 in combinations(TOP_CATS_FOR_NGRAM, 2):
    col_name = f"BG_{c1}_{c2}"
    for df in [train, test]:
        df[col_name] = (df[c1].astype(str) + "_" + df[c2].astype(str)).astype('category')
    BIGRAM_COLS.append(col_name)

print(f"  Created {len(BIGRAM_COLS)} bi-gram features")

# Tri-grams: top 4 cats only to limit = C(4,3) = 4 triples
TOP4 = TOP_CATS_FOR_NGRAM[:4]
print(f"\nCreating Tri-gram features from top 4 categoricals:")
print(f"  Columns: {TOP4}")
for c1, c2, c3 in combinations(TOP4, 3):
    col_name = f"TG_{c1}_{c2}_{c3}"
    for df in [train, test]:
        df[col_name] = (df[c1].astype(str) + "_" + df[c2].astype(str) + "_" + df[c3].astype(str)).astype('category')
    TRIGRAM_COLS.append(col_name)

print(f"  Created {len(TRIGRAM_COLS)} tri-gram features")

NGRAM_COLS = BIGRAM_COLS + TRIGRAM_COLS
print(f"\nTotal N-gram features: {len(NGRAM_COLS)}")

## 5. Primary Model Training Logic

The 15-fold stratified regime ensures that the out-of-fold probability estimates are
sufficiently granular for the ensemble blending stage. Target encoding is handled
within an inner 5-fold loop to prevent information leakage, ensuring that the model
learns generalized categorical dependencies.

In [ ]:
# Feature Summary
FEATURES = NUMS + CATS + NEW_NUMS + NUM_AS_CAT + NGRAM_COLS
TE_COLUMNS = NUM_AS_CAT + CATS
TE_NGRAM_COLUMNS = NGRAM_COLS
TO_REMOVE = NUM_AS_CAT + CATS + NGRAM_COLS
STATS = ['std', 'min', 'max']

print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE")
print("="*60)
print(f"Total Features: {len(FEATURES)}")
print(f"  - Numerical:          {len(NUMS)}")
print(f"  - Categorical:        {len(CATS)}")
print(f"  - Engineered Numerical: {len(NEW_NUMS)}")
print(f"  - Num-as-Cat:         {len(NUM_AS_CAT)}")
print(f"  - Bi-gram:            {len(BIGRAM_COLS)}")
print(f"  - Tri-gram:           {len(TRIGRAM_COLS)}")

In [ ]:
# Model Training with 10-Fold Cross-Validation
print("\n" + "="*60)
print(f"TRAINING WITH {CFG.N_FOLDS}-FOLD CROSS-VALIDATION")
print("="*60)

np.random.seed(CFG.RANDOM_SEED)
skf_outer = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)
skf_inner = StratifiedKFold(n_splits=CFG.INNER_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)

oof = {}
pred = {}
fold_scores = {}
fi = {}
for model_name in models.keys():
    oof[model_name] = np.zeros(len(train))
    pred[model_name] = np.zeros(len(test))
    fold_scores[model_name] = []
    fi[model_name] = pd.DataFrame()

t0 = time.time()

for i, (train_idx, val_idx) in enumerate(skf_outer.split(train, train[CFG.TARGET])):
    print(f"\n{'='*50}")
    print(f"Fold {i+1}/{CFG.N_FOLDS}")
    print(f"{'='*50}")
    
    X_tr  = train.loc[train_idx, FEATURES + [CFG.TARGET]].reset_index(drop=True).copy()
    y_tr  = train.loc[train_idx, CFG.TARGET].values
    X_val = train.loc[val_idx, FEATURES].reset_index(drop=True).copy()
    y_val = train.loc[val_idx, CFG.TARGET].values
    X_te  = test[FEATURES].reset_index(drop=True).copy()
    
    # ─── Inner KFold TE for ORIGINAL categoricals ────────────────────────
    for j, (in_tr, in_va) in enumerate(skf_inner.split(X_tr, y_tr)):
        X_tr2 = X_tr.loc[in_tr, FEATURES + [CFG.TARGET]].copy()
        X_va2 = X_tr.loc[in_va, FEATURES].copy()
        for col in TE_COLUMNS:
            tmp = X_tr2.groupby(col, observed=False)[CFG.TARGET].agg(STATS)
            tmp.columns = [f"TE1_{col}_{s}" for s in STATS]
            X_va2 = X_va2.merge(tmp, on=col, how="left")
            for c in tmp.columns:
                X_tr.loc[in_va, c] = X_va2[c].values.astype("float32")
                
    # Full-fold TE stat for val/test (original cats)
    for col in TE_COLUMNS:
        tmp = X_tr.groupby(col, observed=False)[CFG.TARGET].agg(STATS)
        tmp.columns = [f"TE1_{col}_{s}" for s in STATS]
        tmp = tmp.astype("float32")
        X_val = X_val.merge(tmp, on=col, how="left")
        X_te  = X_te.merge(tmp, on=col, how="left")
        for c in tmp.columns:
            for df in [X_tr, X_val, X_te]:
                df[c] = df[c].fillna(0)
    
    # ─── Inner KFold TE for N-GRAM categoricals ───────────────────────────
    for j, (in_tr, in_va) in enumerate(skf_inner.split(X_tr, y_tr)):
        X_tr2 = X_tr.loc[in_tr].copy()
        X_va2 = X_tr.loc[in_va].copy()
        for col in TE_NGRAM_COLUMNS:
            ng_te = X_tr2.groupby(col, observed=False)[CFG.TARGET].mean()
            ng_name = f"TE_ng_{col}"
            mapped = X_va2[col].astype(str).map(ng_te)
            X_tr.loc[in_va, ng_name] = pd.to_numeric(mapped, errors='coerce').fillna(0.5).astype('float32').values
    
    # Full-fold TE for n-grams on val/test
    for col in TE_NGRAM_COLUMNS:
        ng_te = X_tr.groupby(col, observed=False)[CFG.TARGET].mean()
        ng_name = f"TE_ng_{col}"
        X_val[ng_name] = pd.to_numeric(X_val[col].astype(str).map(ng_te), errors='coerce').fillna(0.5).astype('float32')
        X_te[ng_name]  = pd.to_numeric(X_te[col].astype(str).map(ng_te), errors='coerce').fillna(0.5).astype('float32')
        if ng_name in X_tr.columns:
            X_tr[ng_name] = pd.to_numeric(X_tr[ng_name], errors='coerce').fillna(0.5).astype('float32')
        else:
            X_tr[ng_name] = 0.5
                
    # sklearn TargetEncoder (Mean) for original cats
    TE_MEAN_COLS = [f'TE_{col}' for col in TE_COLUMNS]
    te = TargetEncoder(cv=CFG.INNER_FOLDS, shuffle=True, smooth='auto', target_type='binary', random_state=CFG.RANDOM_SEED)
    X_tr[TE_MEAN_COLS] = te.fit_transform(X_tr[TE_COLUMNS], y_tr)
    X_val[TE_MEAN_COLS] = te.transform(X_val[TE_COLUMNS])
    X_te[TE_MEAN_COLS] = te.transform(X_te[TE_COLUMNS])
    
    # Prepare — remove raw categoricals
    for df in [X_tr, X_val, X_te]:
        for c in CATS + NUM_AS_CAT:
            if c in df.columns:
                df[c] = df[c].astype(str).astype("category")
        df.drop(columns=[c for c in TO_REMOVE if c in df.columns], inplace=True, errors='ignore')
    X_tr.drop(columns=[CFG.TARGET], inplace=True, errors='ignore')
    COLS_XGB = X_tr.columns
    
    if i == 0:
        n_feats = len(COLS_XGB)
        n_ngram_feats = sum(1 for c in COLS_XGB if c.startswith('TE_ng_'))
        print(f"\n  Features for models: {n_feats} ({n_ngram_feats} from n-gram TE)")
    
    # Train models
    for model_name, model in models.items():
        if model_name == 'LGB':
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                      callbacks=[lgb_mod.early_stopping(200, verbose=False), lgb_mod.log_evaluation(0)])
        else:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=1000)
        
        # Store feature importance
        fold_imp = pd.DataFrame({'feature': COLS_XGB, f'importance_fold_{i+1}': model.feature_importances_})
        fi[model_name] = pd.merge(fi[model_name], fold_imp, on='feature', how='outer') if i > 0 else fold_imp
        
        # Record Results
        oof[model_name][val_idx] = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, oof[model_name][val_idx])
        fold_scores[model_name].append(fold_auc)
        
        fold_test_p = model.predict_proba(X_te[COLS_XGB])[:, 1]
        pred[model_name] += fold_test_p / CFG.N_FOLDS
        
        print(f"   Fold {i+1} AUC: {fold_auc:.5f} | Time: {(time.time()-t0)/60:.1f} min")
        del model
    
    del X_tr, X_val, X_te, y_tr, y_val
    gc.collect()

print(f"\n{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")

## 6. Triple Ensemble Blending

Diversity in base learners is critical for ensemble stability. This pipeline combines
three gradient boosting frameworks with distinct splitting algorithms. The final
weights are determined by the COBYLA optimizer, which calculates the optimal balance
between XGBoost, CatBoost, and LightGBM to maximize the cumulative AUC.

In [ ]:
def objective(w):
    oof_ensemble = np.zeros(len(train))
    for i, model_name in enumerate(models.keys()):
        oof_ensemble += oof[model_name] * w[i]
    return -roc_auc_score(train[CFG.TARGET], oof_ensemble)
 
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = Bounds(0, 1)
 
w0 = np.ones(len(models))/len(models)

w = minimize(objective, w0, method='COBYLA', bounds=bounds, constraints=constraints).x

print('Model weights', w)

oof_ensemble = np.zeros(len(train))
pred_ensemble = np.zeros(len(test))
for i, model_name in enumerate(models.keys()):
    oof_ensemble += oof[model_name] * w[i]
    pred_ensemble += pred[model_name] * w[i]

## 7. Performance Diagnostics

Per-fold AUC stability, ROC characteristics, and feature importance rankings
are examined to confirm that the pipeline generalizes consistently and is not
driven by single-fold artifacts.

In [ ]:
# Calculate Overall Metrics
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report

overall_auc = roc_auc_score(train[CFG.TARGET], oof_ensemble)

print("="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"\nEnsemble CV AUC:  {overall_auc:.5f}\n")

for model_name in models.keys():
    print("="*60)
    print(f"Model", model_name)
    overall_auc = roc_auc_score(train[CFG.TARGET], oof[model_name])
    mean_score = np.mean(fold_scores[model_name])
    std_score = np.std(fold_scores[model_name])
    print(f"\nModel CV AUC:  {overall_auc:.5f}")
    print(f"Mean Fold AUC:   {mean_score:.5f} +/- {std_score:.5f}")
    print(f"\nPer-Fold Scores:")
    for i, score in enumerate(fold_scores[model_name]):
        print(f"  Fold {i+1:2d}: {score:.5f}")

In [ ]:
# Visualization: Fold Performance
fig, axes = plt.subplots(len(models), 1, figsize=(10, 5 * len(models)))

for i, model_name in enumerate(models.keys()):
    # 1. Fold AUC Scores
    ax1 = axes[i]
    colors = ['#2ecc71' if s >= mean_score else '#e74c3c' for s in fold_scores[model_name]]
    bars = ax1.bar(range(1, CFG.N_FOLDS + 1), fold_scores[model_name], color=colors, edgecolor='black')
    ax1.axhline(y=mean_score, color='blue', linestyle='--', linewidth=2, label=f'Mean: {mean_score:.5f}')
    ax1.axhline(y=mean_score + std_score, color='blue', linestyle=':', alpha=0.5)
    ax1.axhline(y=mean_score - std_score, color='blue', linestyle=':', alpha=0.5)
    ax1.fill_between(range(0, 12), mean_score - std_score, mean_score + std_score, alpha=0.1, color='blue')
    ax1.set_xlabel('Fold')
    ax1.set_ylabel('AUC Score')
    ax1.set_title('AUC Score by Fold for ' + model_name, fontsize=12, fontweight='bold')
    ax1.set_xticks(range(1, CFG.N_FOLDS + 1))
    ax1.legend()
    ax1.set_ylim([min(fold_scores[model_name]) - 0.01, max(fold_scores[model_name]) + 0.01])
    
    for bar, score in zip(bars, fold_scores[model_name]):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, 
                f'{score:.4f}', ha='center', va='bottom', fontsize=8)

In [ ]:
# Visualization: Fold Performance
fig, axes = plt.subplots(3, 1, figsize=(10, 15))

# 1. ROC Curve
ax2 = axes[0]
fpr, tpr, thresholds = roc_curve(train[CFG.TARGET], oof_ensemble)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='#3498db', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.5f})')
ax2.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
ax2.fill_between(fpr, tpr, alpha=0.2, color='#3498db')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

# 2. Prediction Distribution
ax3 = axes[1]
ax3.hist(oof_ensemble[train[CFG.TARGET] == 0], bins=50, alpha=0.7, label='No Churn', color='#2ecc71', density=True)
ax3.hist(oof_ensemble[train[CFG.TARGET] == 1], bins=50, alpha=0.7, label='Churn', color='#e74c3c', density=True)
ax3.axvline(x=0.5, color='black', linestyle='--', label='Default Threshold')
ax3.set_xlabel('Predicted Probability')
ax3.set_ylabel('Density')
ax3.set_title('OOF Prediction Distribution by True Label', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 3. Precision-Recall Curve
ax4 = axes[2]
precision, recall, _ = precision_recall_curve(train[CFG.TARGET], oof_ensemble)
ax4.plot(recall, precision, color='#9b59b6', linewidth=2)
ax4.fill_between(recall, precision, alpha=0.2, color='#9b59b6')
ax4.set_xlabel('Recall')
ax4.set_ylabel('Precision')
ax4.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for i, model_name in enumerate(models.keys()):
    # Feature Importance Analysis
    # Calculate mean importance across folds
    print("="*60)
    print(f"Model", model_name)
    feature_importances = fi[model_name]
    imp_cols = [c for c in feature_importances.columns if c.startswith('importance_fold')]
    feature_importances['mean_importance'] = feature_importances[imp_cols].mean(axis=1)
    feature_importances['std_importance'] = feature_importances[imp_cols].std(axis=1)
    feature_importances = feature_importances.sort_values('mean_importance', ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Top 20 features overall
    ax1 = axes[0]
    top20 = feature_importances.head(20)
    colors = ['#e74c3c' if 'TE_ng_' in f else '#3498db' for f in top20['feature']]
    bars = ax1.barh(range(len(top20)), top20['mean_importance'], xerr=top20['std_importance'], 
                   color=colors, edgecolor='black', capsize=3)
    ax1.set_yticks(range(len(top20)))
    ax1.set_yticklabels(top20['feature'], fontsize=9)
    ax1.invert_yaxis()
    ax1.set_xlabel('Mean Importance')
    ax1.set_title('Top 20 Features by Importance', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Legend for colors
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#e74c3c', label='N-gram TE Features'),
                       Patch(facecolor='#3498db', label='Other Features')]
    ax1.legend(handles=legend_elements, loc='lower right')
    
    # Top 15 N-gram features
    ax2 = axes[1]
    ngram_imp = feature_importances[feature_importances['feature'].str.startswith('TE_ng_')].head(15)
    bars = ax2.barh(range(len(ngram_imp)), ngram_imp['mean_importance'], 
                   color='#e74c3c', edgecolor='black')
    ax2.set_yticks(range(len(ngram_imp)))
    ax2.set_yticklabels(ngram_imp['feature'], fontsize=9)
    ax2.invert_yaxis()
    ax2.set_xlabel('Mean Importance')
    ax2.set_title('Top 15 N-gram TE Features', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print top N-gram features
    print("\nTop 10 N-gram TE Features by Importance:")
    print("-"*60)
    for i, row in ngram_imp.head(10).iterrows():
        print(f"  {row['feature']:50s} {row['mean_importance']:.4f}")

In [ ]:
# Test Set Prediction Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(pred_ensemble, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold = 0.5')
axes[0].axvline(x=np.mean(pred_ensemble), color='green', linestyle='-', linewidth=2, label=f'Mean = {np.mean(pred_ensemble):.3f}')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Test Set Prediction Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot comparison
axes[1].boxplot([oof_ensemble[train[CFG.TARGET] == 0], oof_ensemble[train[CFG.TARGET] == 1], pred_ensemble],
               labels=['OOF No Churn', 'OOF Churn', 'Test Set'])
axes[1].set_ylabel('Predicted Probability')
axes[1].set_title('Prediction Distribution Comparison', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('test_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTest Set Statistics:")
print(f"  Mean prediction: {np.mean(pred_ensemble):.4f}")
print(f"  Std prediction:  {np.std(pred_ensemble):.4f}")
print(f"  Min prediction:  {np.min(pred_ensemble):.4f}")
print(f"  Max prediction:  {np.max(pred_ensemble):.4f}")
print(f"  Predicted churn rate (threshold=0.5): {(pred_ensemble >= 0.5).mean()*100:.2f}%")

## 8. Submission

The selected ensemble vector is formatted into the required CSV structure.

In [ ]:
# Save OOF Predictions
oof_df = pd.DataFrame({
    'id': train_ids,
    CFG.TARGET: oof_ensemble
})
for model_name in models.keys():
    oof_df[CFG.TARGET + '_' + model_name] = oof[model_name]
oof_df.to_csv('oof_predictions.csv', index=False)
print(f"OOF predictions saved to 'oof_predictions.csv'")
print(f"  Shape: {oof_df.shape}")
display(oof_df.head())

# Save Test Predictions (Submission File)
sub_df = pd.DataFrame({
    'id': test_ids,
    CFG.TARGET: pred_ensemble
})
sub_df.to_csv('submission.csv', index=False)
print(f"\nSubmission saved to 'submission.csv'")
print(f"  Shape: {sub_df.shape}")
display(sub_df.head())

## 9. Analysis Summary

This guide detailed a production-grade approach to binary churn classification:

1. **Data Integration**: Verified historical records from the original Telco dataset were merged with competition data to stabilize learned distributions.
2. **Feature Synthesis**: A 7-stage pipeline generated 121 features across arithmetic, categorical n-gram, distribution-percentile, and digit-level dimensions.
3. **Target Encoding**: Inner 5-fold CV target encoding prevented leakage while preserving categorical discriminative power.
4. **Primary Training**: XGBoost and CatBoost were trained under a 20-fold stratified scheme with GPU-accelerated early stopping.
5. **Stacked Ensembling**: A Ridge meta-learner was trained on out-of-fold prediction vectors, adaptively weighting primary models based on their localized strengths.

---
**Citation:**
Yao Yan, Walter Reade, Elizabeth Park. Predict Customer Churn. https://kaggle.com/competitions/playground-series-s6e3, 2026. Kaggle.